<a href="https://colab.research.google.com/github/spancharapula-byte/physics-informed-ecostress-downscaling/blob/main/src/PINN_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
pinn_model.py

Physics-informed constraint for U-Net LST downscaling.

Uses heat diffusion equation:

    ∇ · (k ∇T) = 0


Inputs:

    predicted_lst:
        U-Net output

        Shape:
        (B,1,H,W)


    conductivity:
        Surface thermal conductivity map

        Shape:
        (B,1,H,W)

Output:

    physics_loss
"""


import torch
import torch.nn as nn
import torch.nn.functional as F



class HeatDiffusionPINN(nn.Module):

    def __init__(
        self,
        dx=30.0,
        dy=30.0
    ):

        super().__init__()

        self.dx = dx
        self.dy = dy



    def gradient_x(self, x):

        """
        Calculates temperature change
        horizontally.
        """

        dx = (
            x[:, :, :, 2:]
            -
            x[:, :, :, :-2]
        ) / (2 * self.dx)


        dx = F.pad(
            dx,
            (1,1,0,0),
            mode="replicate"
        )

        return dx



    def gradient_y(self, x):

        """
        Calculates temperature change
        vertically.
        """

        dy = (
            x[:, :, 2:, :]
            -
            x[:, :, :-2, :]
        ) / (2 * self.dy)


        dy = F.pad(
            dy,
            (0,0,1,1),
            mode="replicate"
        )

        return dy



    def forward(
        self,
        predicted_lst,
        conductivity
    ):

        """
        Connects directly with U-Net output.

        predicted_lst:
            Output from U-Net

        conductivity:
            Land surface conductivity map
        """


        # Make conductivity:
        # (B,H,W) → (B,1,H,W)

        if conductivity.ndim == 3:

            conductivity = conductivity.unsqueeze(1)



        # Resize conductivity to U-Net output

        if conductivity.shape[-2:] != predicted_lst.shape[-2:]:

            conductivity = F.interpolate(
                conductivity,
                size=predicted_lst.shape[-2:],
                mode="nearest"
            )



        conductivity = conductivity.to(
            predicted_lst.device
        )



        # Prevent invalid conductivity

        conductivity = torch.clamp(
            conductivity,
            min=1e-6
        )



        # Temperature gradients

        dT_dx = self.gradient_x(
            predicted_lst
        )


        dT_dy = self.gradient_y(
            predicted_lst
        )



        # Heat flux:

        # q = -k∇T

        flux_x = (
            -conductivity
            *
            dT_dx
        )


        flux_y = (
            -conductivity
            *
            dT_dy
        )



        # Divergence:

        # ∇·q

        div_q = (
            self.gradient_x(flux_x)
            +
            self.gradient_y(flux_y)
        )



        # PDE residual should approach zero

        physics_loss = torch.mean(
            div_q ** 2
        )


        return physics_loss

In [2]:
if __name__ == "__main__":
    B, H, W = 2, 64, 64

    predicted_lst = torch.randn(B, 1, H, W)
    conductivity = torch.ones(B, 1, H, W)

    pinn = HeatDiffusionPINN(dx=30.0, dy=30.0)

    loss = pinn(predicted_lst, conductivity)

    print("Physics loss:", loss.item())

Physics loss: 1.4992166370575433e-06
